In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import pandas as pd

# Binance kline column names
COLUMNS = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

DATA_DIR = Path("../data/raw")

# Find all CSV files
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} files")

dfs = []

for file in csv_files:
    print(f"Loading {file.name}")

    df = pd.read_csv(
        file,
        header=None,
        names=COLUMNS
    )

    dfs.append(df)

# Merge all months
df = pd.concat(dfs, ignore_index=True)

# Keep only required columns
df = df[
    [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
    ]
]

# Convert timestamp
df["open_time"] = pd.to_datetime(
    df["open_time"],
    unit="ms",
    utc=True
)

# Convert numeric columns
numeric_cols = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
    
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col])

# Sort chronologically
df = df.sort_values("open_time")

# Remove duplicates if any
df = df.drop_duplicates(subset=["open_time"])

# Reset index
df = df.reset_index(drop=True)

print("\nDataset Summary")
print(df.info())
print(df.head())
print(df.tail())

# Save parquet
output_path = "data/btcusdt_1h_2023_2026.parquet"

df.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print(f"\nSaved to {output_path}")
print(f"Rows: {len(df):,}")

Found 41 files
Loading BTCUSDT-1h-2023-01.csv
Loading BTCUSDT-1h-2023-02.csv
Loading BTCUSDT-1h-2023-03.csv
Loading BTCUSDT-1h-2023-04.csv
Loading BTCUSDT-1h-2023-05.csv
Loading BTCUSDT-1h-2023-06.csv
Loading BTCUSDT-1h-2023-07.csv
Loading BTCUSDT-1h-2023-08.csv
Loading BTCUSDT-1h-2023-09.csv
Loading BTCUSDT-1h-2023-10.csv
Loading BTCUSDT-1h-2023-11.csv
Loading BTCUSDT-1h-2023-12.csv
Loading BTCUSDT-1h-2024-01.csv
Loading BTCUSDT-1h-2024-02.csv
Loading BTCUSDT-1h-2024-03.csv
Loading BTCUSDT-1h-2024-04.csv
Loading BTCUSDT-1h-2024-05.csv
Loading BTCUSDT-1h-2024-06.csv
Loading BTCUSDT-1h-2024-07.csv
Loading BTCUSDT-1h-2024-08.csv
Loading BTCUSDT-1h-2024-09.csv
Loading BTCUSDT-1h-2024-10.csv
Loading BTCUSDT-1h-2024-11.csv
Loading BTCUSDT-1h-2024-12.csv
Loading BTCUSDT-1h-2025-01.csv
Loading BTCUSDT-1h-2025-02.csv
Loading BTCUSDT-1h-2025-03.csv
Loading BTCUSDT-1h-2025-04.csv
Loading BTCUSDT-1h-2025-05.csv
Loading BTCUSDT-1h-2025-06.csv
Loading BTCUSDT-1h-2025-07.csv
Loading BTCUSDT-1h-2025-

OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 56971-10-25 00:00:00

In [3]:
print(df["open_time"].head())
print(df["open_time"].tail())
print(df["open_time"].dtype)

0    1672531200000
1    1672534800000
2    1672538400000
3    1672542000000
4    1672545600000
Name: open_time, dtype: int64
29922    1780254000000000
29923    1780257600000000
29924    1780261200000000
29925    1780264800000000
29926    1780268400000000
Name: open_time, dtype: int64
int64


In [4]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1672531200000,16541.77,16545.70,16508.39,16529.67,4364.83570,1672534799999,7.214629e+07,149854,2179.94772,3.603235e+07,0
1,1672534800000,16529.59,16556.80,16525.78,16551.47,3590.06669,1672538399999,5.937676e+07,126556,1730.24901,2.861742e+07,0
2,1672538400000,16551.47,16559.77,16538.14,16548.19,3318.84038,1672541999999,5.491945e+07,115398,1611.12302,2.666087e+07,0
3,1672542000000,16548.19,16548.19,16518.21,16533.04,4242.08050,1672545599999,7.012254e+07,137724,2096.09287,3.464904e+07,0
4,1672545600000,16533.04,16535.97,16511.92,16521.85,4285.00909,1672549199999,7.080264e+07,129535,2188.40175,3.615982e+07,0


In [5]:
print(df["open_time"].astype(str).str.len().value_counts())

open_time
13    17543
16    12384
Name: count, dtype: int64


In [6]:
lengths = df["open_time"].astype(str).str.len()

change_points = lengths.ne(lengths.shift())

df.loc[change_points, ["open_time"]]

,open_time
0,1672531200000
17543,1735689600000000


In [7]:
for file in csv_files:
    tmp = pd.read_csv(file, header=None)

    print(f"\n{file.name}")
    print(tmp.dtypes)
    print(tmp.iloc[0, 0], tmp.iloc[-1, 0])


BTCUSDT-1h-2023-01.csv
0       int64
1     float64
2     float64
3     float64
4     float64
5     float64
6       int64
7     float64
8       int64
9     float64
10    float64
11      int64
dtype: object
1672531200000 1675206000000

BTCUSDT-1h-2023-02.csv
0       int64
1     float64
2     float64
3     float64
4     float64
5     float64
6       int64
7     float64
8       int64
9     float64
10    float64
11      int64
dtype: object
1675209600000 1677625200000

BTCUSDT-1h-2023-03.csv
0       int64
1     float64
2     float64
3     float64
4     float64
5     float64
6       int64
7     float64
8       int64
9     float64
10    float64
11      int64
dtype: object
1677628800000 1680303600000

BTCUSDT-1h-2023-04.csv
0       int64
1     float64
2     float64
3     float64
4     float64
5     float64
6       int64
7     float64
8       int64
9     float64
10    float64
11      int64
dtype: object
1680307200000 1682895600000

BTCUSDT-1h-2023-05.csv
0       int64
1     float64
2     float6

In [10]:
#Normalize
def normalize_timestamp(ts):
    ts = int(ts)

    if len(str(ts)) == 16:
        return ts // 1000

    return ts

In [11]:
df["open_time"] = df["open_time"].apply(normalize_timestamp)

df["open_time"] = pd.to_datetime(
    df["open_time"],
    unit="ms",
    utc=True
)

In [12]:
print(df["open_time"].astype(str).str.len().value_counts())

open_time
25    29927
Name: count, dtype: int64


In [15]:
diffs = df["open_time"].diff().value_counts()

print(diffs.head())

open_time
0 days 01:00:00    29925
0 days 02:00:00        1
Name: count, dtype: int64


In [17]:
import pandas as pd

# ==========================================
# 1. Convert to datetime
# ==========================================

df["open_time"] = pd.to_datetime(
    df["open_time"],
    unit="ms",
    utc=True
)

# ==========================================
# 2. Sort chronologically
# ==========================================

df = df.sort_values("open_time")

# ==========================================
# 3. Remove duplicates
# ==========================================

before = len(df)

df = df.drop_duplicates(
    subset=["open_time"]
)

after = len(df)

print(f"Removed {before - after} duplicate candles")

# ==========================================
# 4. Reset index
# ==========================================

df = df.reset_index(drop=True)

# ==========================================
# 5. Verify continuity
# ==========================================

expected = pd.date_range(
    start=df["open_time"].min(),
    end=df["open_time"].max(),
    freq="1h",
    tz="UTC"
)

missing = expected.difference(df["open_time"])

print(f"Missing candles: {len(missing)}")

if len(missing) > 0:
    print("\nFirst 10 missing candles:")
    print(missing[:10])

# ==========================================
# 6. Dataset Summary
# ==========================================

print("\nDataset Summary")
print("-" * 50)

print("Start :", df["open_time"].min())
print("End   :", df["open_time"].max())
print("Rows  :", len(df))

# ==========================================
# 7. Save Parquet
# ==========================================

df.to_parquet(
    "../data/cleaned/btcusdt_1h_clean.parquet",
    index=False
)

print("\nSaved -> btcusdt_1h_clean.parquet")

Removed 0 duplicate candles
Missing candles: 1

First 10 missing candles:
DatetimeIndex(['2023-03-24 13:00:00+00:00'], dtype='datetime64[ns, UTC]', freq='h')

Dataset Summary
--------------------------------------------------
Start : 2023-01-01 00:00:00+00:00
End   : 2026-05-31 23:00:00+00:00
Rows  : 29927

Saved -> btcusdt_1h_clean.parquet
